In [1]:
import sys, os, site
print("Notebook python:", sys.executable)
print("User:", os.environ.get("USER"))
print("JUPYTER_PATH:", os.environ.get("JUPYTER_PATH"))
print("site.USER_BASE:", site.USER_BASE)

Notebook python: /home/bosho/.conda/envs/fedgnn/bin/python
User: bosho
JUPYTER_PATH: None
site.USER_BASE: /home/bosho/.local


In [2]:
import subprocess
import textwrap


In [3]:
def run(cmd: str) -> None:
    print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout.rstrip())
    if p.stderr:
        print(p.stderr.rstrip())
    print(f"[exit_code={p.returncode}]")

print("=== GPU Diagnostic (Notebook) ===")

=== GPU Diagnostic (Notebook) ===


In [4]:
# 1) nvidia-smi (driver + vGPU visibility)
run("nvidia-smi")


$ nvidia-smi
Mon Apr 27 12:20:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40-48Q                 On  |   00000000:02:00.0 Off |                    0 |
| N/A   N/A    P8             N/A /  N/A  |      52MiB /  49152MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------------------

In [5]:

# 2) PyTorch CUDA check + simple CUDA compute
import torch

In [6]:
print("\n=== PyTorch / CUDA ===")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA compiled: {torch.version.cuda}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.cuda.device_count(): {torch.cuda.device_count()}")


=== PyTorch / CUDA ===
PyTorch: 2.4.1+cu121
CUDA compiled: 12.1
torch.cuda.is_available(): True
torch.cuda.device_count(): 1


In [8]:
# lets multiply two tensors with torch
t = torch.tensor([1.0], device="cuda")
r = t * 2
r

tensor([2.], device='cuda:0')

In [9]:

torch.cuda.synchronize()
print("RESULT: GPU fully functional")
print("Compute check:", r.item())



RESULT: GPU fully functional
Compute check: 2.0


In [7]:
if torch.cuda.is_available() and torch.cuda.device_count() > 0:
    try:
        print(f"Device name: {torch.cuda.get_device_name(0)}")
        props = torch.cuda.get_device_properties(0)
        print(f"VRAM: {props.total_memory / 1e9:.1f} GB")

        t = torch.tensor([1.0], device="cuda")
        r = t * 2
        torch.cuda.synchronize()
        print("RESULT: GPU fully functional")
        print("Compute check:", r.item())
    except RuntimeError as e:
        print("RESULT: GPU compute FAILED")
        print("Error:", e)
else:
    print("RESULT: CUDA not available to PyTorch (no usable GPU detected)")

Device name: NVIDIA L40-48Q
VRAM: 51.2 GB
RESULT: GPU compute FAILED
Error: CUDA error: operation not supported
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.



In [10]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

Torch: 2.4.1+cu121
CUDA available: True
Torch CUDA: 12.1
GPU count: 1
GPU: NVIDIA L40-48Q
Capability: (8, 9)


In [11]:
import torch

x = torch.empty(1, device="cuda")
print(x)
torch.cuda.synchronize()
print("GPU works")

tensor([2.], device='cuda:0')
GPU works
